# Baseline and Enhanced Logistic Regression Model

## Business problem

The ecommerce platform wants to identify sessions that are likely to result in purchase activity. Purchase sessions are rare, so the business needs a model that can help prioritize high-intent sessions for actions such as remarketing, saved-cart reminders, personalized recommendations, or targeted offers.

## Objective

Build an interpretable session-level Logistic Regression model to predict whether a completed session contains purchase activity.

Target variable:

- `purchased`: `1` if the session contains a transaction event, otherwise `0`

## Data source

This notebook now loads the enhanced Tableau-ready session-level dataset:

```text
Dataset/Processed Dataset/tableau_session_funnel_data.csv
```

That file is created in:

```text
Notebooks/Tableau Data Preparation.ipynb
```

The Tableau-ready dataset contains both the original session funnel columns and the reusable enhanced features.

## Modeling flow

This notebook still starts with a simple baseline model for comparison, then builds the enhanced model.

Baseline features:

- `viewed`
- `carted`
- `session_duration_mins`

Enhanced features:

- `viewed`
- `carted`
- `log_session_duration_mins`
- `visitor_session_count`
- `is_repeat_visitor`
- `session_start_hour`
- `session_start_dayofweek`
- `is_weekend`

## Important modeling note

`event_count` is excluded from the model feature sets because it is calculated after the full session is complete and may include the transaction event itself.

This baseline model uses completed session-level features. That means it is useful for understanding predictive signals and creating a benchmark, but it is not yet a production real-time model. A production model would need features available at the moment of prediction, such as after the first event, after the first few events, or before checkout.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load enhanced Tableau-ready session-level dataset
# This file is generated from funnel_data.csv in Notebooks/Tableau Data Preparation.ipynb.
from pathlib import Path

tableau_input_path = Path('../Dataset/Processed Dataset/tableau_session_funnel_data.csv')

if not tableau_input_path.exists():
    raise FileNotFoundError(
        "Input file not found: '../Dataset/Processed Dataset/tableau_session_funnel_data.csv'. "
        "Please run 'Notebooks/Tableau Data Preparation.ipynb' first to generate and save the Tableau-ready dataset. "
        "If funnel_data.csv is also missing, run 'Notebooks/Funnel Analytics.ipynb' before Tableau Data Preparation."
    )

funnel_df = pd.read_csv(tableau_input_path)

In [ ]:
funnel_df.head()

In [ ]:
funnel_df.shape

In [ ]:
funnel_df.session_key.nunique()

In [ ]:
funnel_df.isnull().sum()

In [ ]:
funnel_df.info()

In [ ]:
funnel_df.describe(include='all')

In [ ]:
funnel_df.purchased.value_counts()

In [ ]:
# Target variable distribution
purchase_counts = (
    funnel_df['purchased']
    .value_counts()
    .reindex([0, 1])
    .reset_index()
)
purchase_counts.columns = ['purchased', 'count']
purchase_counts['label'] = purchase_counts['purchased'].map({0: 'No purchase', 1: 'Purchase'})

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(
    data=purchase_counts,
    x='label',
    y='count',
    hue='label',
    palette=['#F58518', '#54A24B'],
    legend=False,
    ax=ax
)

ax.set_title('Target Variable Distribution: Purchase vs No Purchase', fontsize=13, pad=12)
ax.set_xlabel('Target class')
ax.set_ylabel('Number of sessions')
ax.ticklabel_format(style='plain', axis='y')

for patch, count in zip(ax.patches, purchase_counts['count']):
    x = patch.get_x() + patch.get_width() / 2
    y = patch.get_height()
    ax.text(
        x,
        y + (purchase_counts['count'].max() * 0.015),
        f'{count:,.0f}',
        ha='center',
        va='bottom',
        fontsize=10,
        fontweight='bold'
    )

ax.set_ylim(0, purchase_counts['count'].max() * 1.12)
sns.despine()
plt.tight_layout()
plt.show()


# Data is highly imbalanced 

In [ ]:
funnel_df.columns

In [ ]:
# Baseline feature set
# The dataset contains enhanced features, but this first model intentionally uses
# only the original simple session-level features for comparison.
# Excluding event_count because it may include the transaction event itself.
X = funnel_df[['viewed', 'carted', 'session_duration_mins']]
y = funnel_df['purchased']


In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
funnel_df['purchased'].value_counts(normalize=True)

# Building the Logistic Regression Model¶

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
#from sklearn.metrics import accuracy_score
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    confusion_matrix, 
    classification_report, 
    roc_auc_score, 
    log_loss
)

In [ ]:
# Split data into train and test dataset
xtrain,xtest,ytrain,ytest=train_test_split(X,y,test_size=0.2,random_state=15)

In [ ]:
# Build Logistic Regression Model
LF=LogisticRegression()
LF.fit(xtrain,ytrain)
# Training data
ypredtrain=LF.predict(xtrain)
# Testing data
ypredtest=LF.predict(xtest) ## Discrete classes (0 or 1)
yprobtest = LF.predict_proba(xtest) # Probabilities for log_loss and ROC-AUC

In [ ]:
print("Training accuracy")
print("------------")
print(accuracy_score(ytrain,ypredtrain))
print("Testing accuracy")
print("------------")
print(accuracy_score(ytest,ypredtest))

In [ ]:
accuracy = accuracy_score(ytest, ypredtest)
conf_matrix = confusion_matrix(ytest, ypredtest)
precision = precision_score(ytest, ypredtest)
recall = recall_score(ytest, ypredtest)
class_report = classification_report(ytest, ypredtest)
roc_auc = roc_auc_score(ytest, yprobtest[:, 1])
loss = log_loss(ytest, yprobtest)

In [ ]:
# 4. Print results
print(f"Accuracy: {accuracy:.4f}")
print(f"ROC-AUC Score: {roc_auc:.4f}")
print(f"Log Loss: {loss:.4f}")
print("\nConfusion Matrix:\n", conf_matrix)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print("\nClassification Report:\n", class_report)

* Precision for purchase: 68.06%
* Recall for purchase:    13.44%
* ROC-AUC:                98.23%

## ROC-AUC Curve

ROC-AUC evaluates how well the model separates purchase sessions from non-purchase sessions across different classification thresholds.

- ROC curve plots True Positive Rate against False Positive Rate.
- AUC summarizes the curve into one number.
- AUC closer to `1.0` means stronger class separation.
- AUC around `0.5` means the model is close to random guessing.

For this imbalanced dataset, ROC-AUC is more useful than accuracy, but it should still be reviewed together with precision, recall, and PR-AUC.


In [ ]:
# Plot ROC-AUC curve
from sklearn.metrics import roc_curve, auc

# Probability of the positive class: purchased = 1
y_score = yprobtest[:, 1]

fpr, tpr, thresholds = roc_curve(ytest, y_score)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    fpr,
    tpr,
    color='#4C78A8',
    linewidth=2,
    label=f'Logistic Regression (AUC = {roc_auc:.4f})'
)
ax.plot(
    [0, 1],
    [0, 1],
    color='#E45756',
    linestyle='--',
    linewidth=1.5,
    label='Random classifier'
)

ax.set_title('ROC Curve - Logistic Regression')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate / Recall')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


Interpretation:

The ROC curve shows the trade-off between catching purchase sessions and incorrectly flagging non-purchase sessions as purchases. A high ROC-AUC means the model ranks purchase sessions higher than non-purchase sessions well overall.

However, because purchase sessions are rare, ROC-AUC alone is not enough. The model should also be evaluated using recall, precision, and preferably PR-AUC to understand how well it identifies the minority purchase class.


## Logistic Regression Interpretation So Far

The logistic regression model is a baseline model for predicting whether a session contains purchase activity.

### 1. Target variable is highly imbalanced

The target variable has a very large class imbalance:

- Non-purchase sessions: about `99.19%`
- Purchase sessions: about `0.81%`

Because of this imbalance, accuracy alone is not a reliable metric. A model can achieve high accuracy by predicting most sessions as non-purchase.

### 2. Accuracy is high, but it is misleading

The model achieved around `99.24%` test accuracy. However, this high accuracy is mainly driven by the majority class, which is non-purchase sessions.

Key interpretation:

> Accuracy looks strong, but it does not tell us whether the model is good at identifying purchase sessions.

### 3. Purchase recall is low

The model's recall for purchase sessions is about `13.44%` at the default threshold of `0.5`.

This means:

> Out of all actual purchase sessions, the model catches only about 13%.

So the model is conservative. It predicts purchase only when it is very confident, but it misses many actual purchase sessions.

### 4. Precision is stronger than recall

Purchase precision is about `68.06%`.

This means:

> When the model predicts purchase, it is often correct. However, it predicts purchase for only a small number of sessions.

### 5. ROC-AUC is high

ROC-AUC is around `0.9823`, which means the model ranks purchase sessions higher than non-purchase sessions very well overall.

Interpretation:

> If we randomly compare one purchase session and one non-purchase session, the model is very likely to assign a higher purchase probability to the purchase session.

However, high ROC-AUC does not mean the default `0.5` threshold is good. The model can rank sessions well while still missing many purchases at the default threshold.

### 6. False Positive Rate at threshold 0.5 is very low

From the confusion matrix, the false positive rate at threshold `0.5` is approximately `0.0524%`.

This means:

> The model rarely misclassifies non-purchase sessions as purchase.

But the trade-off is low recall: it also misses many actual purchase sessions.

### Current conclusion

The baseline logistic regression model has strong ranking ability but weak purchase detection at the default threshold.

Business interpretation:

> The model can separate likely purchase sessions from non-purchase sessions, but the default threshold is too conservative for identifying purchase sessions. For business use cases like lead scoring, remarketing, or purchase propensity targeting, recall and probability ranking are more important than accuracy alone.

### Recommended next steps

- Add stratified train/test split using `stratify=y`.
- Use `class_weight='balanced'` to handle class imbalance.
- Scale numeric features using `StandardScaler`.
- Evaluate PR-AUC and the Precision-Recall curve.
- Tune the classification threshold based on the business goal.


In [ ]:
X.head()

## Logistic Regression with Class Weight Balanced

The base logistic regression model has high accuracy but low recall for purchase sessions because the target variable is highly imbalanced.

To handle class imbalance, train another logistic regression model using:

`class_weight='balanced'`

This gives more weight to the minority class (`purchased = 1`) during training.

Goal:

> Improve recall for purchase sessions, even if precision or accuracy decreases.


In [ ]:
# Build Logistic Regression Model with class_weight='balanced'
LR_balanced = LogisticRegression(class_weight='balanced', max_iter=1000)
LR_balanced.fit(xtrain, ytrain)

# Training predictions
ypredtrain_balanced = LR_balanced.predict(xtrain)

# Testing predictions
ypredtest_balanced = LR_balanced.predict(xtest)
yprobtest_balanced = LR_balanced.predict_proba(xtest)


In [ ]:
# Accuracy comparison for balanced model
print("Balanced Logistic Regression - Training accuracy")
print("------------")
print(accuracy_score(ytrain, ypredtrain_balanced))

print("Balanced Logistic Regression - Testing accuracy")
print("------------")
print(accuracy_score(ytest, ypredtest_balanced))


In [ ]:
# Evaluation metrics for balanced logistic regression
balanced_accuracy = accuracy_score(ytest, ypredtest_balanced)
balanced_conf_matrix = confusion_matrix(ytest, ypredtest_balanced)
balanced_precision = precision_score(ytest, ypredtest_balanced)
balanced_recall = recall_score(ytest, ypredtest_balanced)
balanced_class_report = classification_report(ytest, ypredtest_balanced)
balanced_roc_auc = roc_auc_score(ytest, yprobtest_balanced[:, 1])
balanced_loss = log_loss(ytest, yprobtest_balanced)


In [ ]:
# Print balanced model results
print(f"Accuracy: {balanced_accuracy:.4f}")
print(f"ROC-AUC Score: {balanced_roc_auc:.4f}")
print(f"Log Loss: {balanced_loss:.4f}")
print("\nConfusion Matrix:\n", balanced_conf_matrix)
print(f"Precision: {balanced_precision:.4f}")
print(f"Recall:    {balanced_recall:.4f}")
print("\nClassification Report:\n", balanced_class_report)


Interpretation guide:

Compare this balanced model with the base logistic regression model.

Expected trade-off:

- Recall for purchase sessions should increase.
- Precision may decrease.
- Accuracy may decrease because the model predicts more purchase cases.

For this project, higher recall may be useful if the business goal is to identify more likely purchase sessions for targeting or intervention. However, the final threshold should still be tuned based on the business cost of false positives vs false negatives.


## Base vs Balanced Logistic Regression Comparison

Compare the base logistic regression model with the balanced logistic regression model to understand the precision-recall trade-off.


In [ ]:
# Compare base and balanced logistic regression models
model_comparison = pd.DataFrame({
    'model': ['Base Logistic Regression', 'Balanced Logistic Regression'],
    'accuracy': [accuracy, balanced_accuracy],
    'roc_auc': [roc_auc, balanced_roc_auc],
    'log_loss': [loss, balanced_loss],
    'precision_purchase': [precision, balanced_precision],
    'recall_purchase': [recall, balanced_recall],
    'true_positives': [conf_matrix[1, 1], balanced_conf_matrix[1, 1]],
    'false_positives': [conf_matrix[0, 1], balanced_conf_matrix[0, 1]],
    'false_negatives': [conf_matrix[1, 0], balanced_conf_matrix[1, 0]]
})

model_comparison_display = model_comparison.copy()

for col in ['accuracy', 'roc_auc', 'precision_purchase', 'recall_purchase']:
    model_comparison_display[col] = model_comparison_display[col].map(lambda x: f'{x:.2%}')

model_comparison_display['log_loss'] = model_comparison_display['log_loss'].map(lambda x: f'{x:.4f}')

model_comparison_display


### Which model is better to use when?

Neither model is universally better. The better choice depends on the business goal.

## Use the base logistic regression model when precision matters more

The base model has higher purchase precision but much lower recall.

Use it when:

- false positives are expensive
- the business only wants to target sessions with very high purchase confidence
- outreach, discounting, or intervention has a meaningful cost
- it is acceptable to miss many purchase sessions as long as predicted purchase sessions are more reliable

Business example:

> Use the base model if sending an offer is costly and the business only wants to target the strongest purchase-propensity sessions.

## Use the balanced logistic regression model when recall matters more

The balanced model has much higher purchase recall but lower precision.

Use it when:

- missing purchase-intent sessions is costly
- the business wants to catch as many likely purchase sessions as possible
- false positives are acceptable or inexpensive
- the model is used for broad remarketing, lead scoring, or user prioritization

Business example:

> Use the balanced model if the goal is to identify more potential buyers for remarketing or personalized follow-up, even if some flagged sessions do not purchase.

## Current recommendation

For this project, the balanced model is more useful for purchase-propensity targeting because the original target class is highly imbalanced and the base model misses most purchase sessions.

However, the final production choice should not rely only on `class_weight='balanced'`. The next step should be threshold tuning and Precision-Recall analysis to choose a decision threshold based on business cost and benefit.


## Balanced Logistic Regression with StandardScaler

Logistic regression can benefit from feature scaling because numeric features are on different scales.

Current features include:

- binary features: `viewed`, `carted`
- continuous/count features: `session_duration_mins`, `event_count`

Use a pipeline so that `StandardScaler` is fit only on the training data and then applied to the test data. This avoids data leakage.


In [ ]:
# Build Balanced Logistic Regression with StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

LR_scaled_balanced = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000))
])

LR_scaled_balanced.fit(xtrain, ytrain)

# Training predictions
ypredtrain_scaled_balanced = LR_scaled_balanced.predict(xtrain)

# Testing predictions
ypredtest_scaled_balanced = LR_scaled_balanced.predict(xtest)
yprobtest_scaled_balanced = LR_scaled_balanced.predict_proba(xtest)


In [ ]:
# Accuracy comparison for scaled balanced model
print("Scaled Balanced Logistic Regression - Training accuracy")
print("------------")
print(accuracy_score(ytrain, ypredtrain_scaled_balanced))

print("Scaled Balanced Logistic Regression - Testing accuracy")
print("------------")
print(accuracy_score(ytest, ypredtest_scaled_balanced))


In [ ]:
# Evaluation metrics for scaled balanced logistic regression
scaled_balanced_accuracy = accuracy_score(ytest, ypredtest_scaled_balanced)
scaled_balanced_conf_matrix = confusion_matrix(ytest, ypredtest_scaled_balanced)
scaled_balanced_precision = precision_score(ytest, ypredtest_scaled_balanced)
scaled_balanced_recall = recall_score(ytest, ypredtest_scaled_balanced)
scaled_balanced_class_report = classification_report(ytest, ypredtest_scaled_balanced)
scaled_balanced_roc_auc = roc_auc_score(ytest, yprobtest_scaled_balanced[:, 1])
scaled_balanced_loss = log_loss(ytest, yprobtest_scaled_balanced)


In [ ]:
# Print scaled balanced model results
print(f"Accuracy: {scaled_balanced_accuracy:.4f}")
print(f"ROC-AUC Score: {scaled_balanced_roc_auc:.4f}")
print(f"Log Loss: {scaled_balanced_loss:.4f}")
print("\nConfusion Matrix:\n", scaled_balanced_conf_matrix)
print(f"Precision: {scaled_balanced_precision:.4f}")
print(f"Recall:    {scaled_balanced_recall:.4f}")
print("\nClassification Report:\n", scaled_balanced_class_report)


Interpretation guide:

Compare this scaled balanced model with the previous balanced model.

Scaling may improve training stability and make regularization behave more fairly across features, but it may not always produce a large metric change. If results are similar, the scaled pipeline is still preferred because it is better modeling practice for logistic regression.


## Precision-Recall Curve and PR-AUC

The dataset is highly imbalanced, with purchase sessions representing a very small share of all sessions. In this case, the Precision-Recall curve is especially useful because it focuses on the positive class: `purchased = 1`.

Precision answers:

> Out of the sessions predicted as purchase, how many were actually purchase?

Recall answers:

> Out of all actual purchase sessions, how many did the model catch?

PR-AUC summarizes the Precision-Recall curve. A higher PR-AUC means the model is better at identifying purchase sessions across different thresholds.


In [ ]:
# Precision-Recall Curve and PR-AUC for the scaled balanced logistic regression model
from sklearn.metrics import precision_recall_curve, average_precision_score

# Probability of the positive class: purchased = 1
y_score_pr = yprobtest_scaled_balanced[:, 1]

precision_vals, recall_vals, pr_thresholds = precision_recall_curve(ytest, y_score_pr)
pr_auc = average_precision_score(ytest, y_score_pr)

baseline_purchase_rate = ytest.mean()

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    recall_vals,
    precision_vals,
    color='#4C78A8',
    linewidth=2,
    label=f'Scaled Balanced LR (PR-AUC = {pr_auc:.4f})'
)
ax.axhline(
    baseline_purchase_rate,
    color='#E45756',
    linestyle='--',
    linewidth=1.5,
    label=f'Baseline purchase rate = {baseline_purchase_rate:.4%}'
)

ax.set_title('Precision-Recall Curve - Scaled Balanced Logistic Regression')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"PR-AUC / Average Precision: {pr_auc:.4f}")
print(f"Baseline purchase rate: {baseline_purchase_rate:.4%}")


Interpretation guide:

The red dashed line shows the baseline purchase rate in the test set. A useful model should have a Precision-Recall curve above this baseline.

For this project, PR-AUC is important because purchase sessions are rare. It gives a better view of how well the model identifies the minority purchase class than accuracy alone.

Business interpretation:

> The Precision-Recall curve helps choose a probability threshold depending on whether the business cares more about precision or recall. If the goal is to catch more likely purchase sessions for remarketing, recall may be prioritized. If outreach is costly, precision may be prioritized.


## Logistic Regression with Minority-Class Oversampling

Another way to handle class imbalance is to oversample the minority class in the training data.

Important:

> Oversampling should be applied only to the training set, not the test set.

This keeps the test set realistic and avoids data leakage.

In this section, purchase sessions in the training data are randomly duplicated until the minority and majority classes are balanced. Then a logistic regression model is trained on the oversampled training data.


In [ ]:
# Oversample the minority class in the training data only
from sklearn.utils import resample

train_data = xtrain.copy()
train_data['purchased'] = ytrain.values

majority_class = train_data[train_data['purchased'] == 0]
minority_class = train_data[train_data['purchased'] == 1]

minority_oversampled = resample(
    minority_class,
    replace=True,
    n_samples=len(majority_class),
    random_state=15
)

train_oversampled = pd.concat([majority_class, minority_oversampled])
train_oversampled = train_oversampled.sample(frac=1, random_state=15).reset_index(drop=True)

xtrain_oversampled = train_oversampled.drop(columns='purchased')
ytrain_oversampled = train_oversampled['purchased']

print("Original training target distribution:")
print(ytrain.value_counts())
print("\nOversampled training target distribution:")
print(ytrain_oversampled.value_counts())


In [ ]:
# Build Logistic Regression model on oversampled training data
LR_oversampled = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

LR_oversampled.fit(xtrain_oversampled, ytrain_oversampled)

# Training predictions on oversampled training data
ypredtrain_oversampled = LR_oversampled.predict(xtrain_oversampled)

# Testing predictions on original untouched test data
ypredtest_oversampled = LR_oversampled.predict(xtest)
yprobtest_oversampled = LR_oversampled.predict_proba(xtest)


In [ ]:
# Accuracy comparison for oversampled model
print("Oversampled Logistic Regression - Training accuracy")
print("------------")
print(accuracy_score(ytrain_oversampled, ypredtrain_oversampled))

print("Oversampled Logistic Regression - Testing accuracy")
print("------------")
print(accuracy_score(ytest, ypredtest_oversampled))


In [ ]:
# Evaluation metrics for oversampled logistic regression
oversampled_accuracy = accuracy_score(ytest, ypredtest_oversampled)
oversampled_conf_matrix = confusion_matrix(ytest, ypredtest_oversampled)
oversampled_precision = precision_score(ytest, ypredtest_oversampled)
oversampled_recall = recall_score(ytest, ypredtest_oversampled)
oversampled_class_report = classification_report(ytest, ypredtest_oversampled)
oversampled_roc_auc = roc_auc_score(ytest, yprobtest_oversampled[:, 1])
oversampled_loss = log_loss(ytest, yprobtest_oversampled)


In [ ]:
# Print oversampled model results
print(f"Accuracy: {oversampled_accuracy:.4f}")
print(f"ROC-AUC Score: {oversampled_roc_auc:.4f}")
print(f"Log Loss: {oversampled_loss:.4f}")
print("\nConfusion Matrix:\n", oversampled_conf_matrix)
print(f"Precision: {oversampled_precision:.4f}")
print(f"Recall:    {oversampled_recall:.4f}")
print("\nClassification Report:\n", oversampled_class_report)


Interpretation guide:

Oversampling usually increases recall for the minority purchase class because the model sees many more purchase examples during training.

Expected trade-off:

- Recall may increase.
- Precision may decrease.
- False positives may increase.
- Accuracy may decrease compared with the base model.

Compare this model with the balanced logistic regression model. If both perform similarly, `class_weight='balanced'` may be simpler because it avoids physically duplicating rows. If oversampling improves recall or PR-AUC meaningfully, it may be useful for the purchase-propensity use case.


## Threshold Tuning for Scaled Balanced Logistic Regression

The default classification threshold is `0.5`:

- predicted probability >= 0.5 -> predict purchase
- predicted probability < 0.5 -> predict no purchase

For imbalanced datasets, `0.5` is not always the best threshold. Threshold tuning helps choose a better trade-off between precision and recall based on the business goal.

Business framing:

- Lower threshold: catches more purchase sessions, but creates more false positives.
- Higher threshold: fewer false positives, but misses more purchase sessions.


In [ ]:
# Threshold tuning for scaled balanced logistic regression
from sklearn.metrics import f1_score

thresholds_to_test = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
threshold_results = []

y_score_threshold = yprobtest_scaled_balanced[:, 1]

for threshold in thresholds_to_test:
    ypred_threshold = (y_score_threshold >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(ytest, ypred_threshold).ravel()

    threshold_results.append({
        'threshold': threshold,
        'accuracy': accuracy_score(ytest, ypred_threshold),
        'precision': precision_score(ytest, ypred_threshold, zero_division=0),
        'recall': recall_score(ytest, ypred_threshold),
        'f1_score': f1_score(ytest, ypred_threshold),
        'true_positives': tp,
        'false_positives': fp,
        'false_negatives': fn,
        'true_negatives': tn
    })

threshold_tuning_results = pd.DataFrame(threshold_results)

threshold_tuning_display = threshold_tuning_results.copy()
for col in ['accuracy', 'precision', 'recall', 'f1_score']:
    threshold_tuning_display[col] = threshold_tuning_display[col].map(lambda x: f'{x:.2%}')

threshold_tuning_display


In [ ]:
# Visualize precision-recall trade-off by threshold
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    threshold_tuning_results['threshold'],
    threshold_tuning_results['precision'],
    marker='o',
    label='Precision',
    color='#4C78A8'
)
ax.plot(
    threshold_tuning_results['threshold'],
    threshold_tuning_results['recall'],
    marker='o',
    label='Recall',
    color='#F58518'
)
ax.plot(
    threshold_tuning_results['threshold'],
    threshold_tuning_results['f1_score'],
    marker='o',
    label='F1-score',
    color='#54A24B'
)

ax.set_title('Precision, Recall, and F1-score by Threshold')
ax.set_xlabel('Classification threshold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Select the threshold with the highest F1-score
best_f1_row = threshold_tuning_results.loc[
    threshold_tuning_results['f1_score'].idxmax()
]

print(f"Best threshold by F1-score: {best_f1_row['threshold']:.2f}")
print(f"Precision: {best_f1_row['precision']:.2%}")
print(f"Recall: {best_f1_row['recall']:.2%}")
print(f"F1-score: {best_f1_row['f1_score']:.2%}")
print(f"False positives: {best_f1_row['false_positives']:,.0f}")
print(f"False negatives: {best_f1_row['false_negatives']:,.0f}")


Interpretation guide:

Threshold tuning shows how model behavior changes as the purchase probability cutoff changes.

- If the business wants to catch more purchase sessions, choose a lower threshold with higher recall.
- If the business wants fewer false positives, choose a higher threshold with higher precision.
- If the business wants a balance between precision and recall, choose the threshold with the best F1-score.

For purchase propensity use cases, the best threshold should be selected based on business costs:

- cost of contacting or targeting a non-purchase session
- value of catching an actual purchase-intent session
- acceptable false positive volume


In [ ]:
funnel_df.head()

In [ ]:
X.head()

## Feature Importance from Logistic Regression Coefficients

For logistic regression, feature importance can be interpreted using model coefficients.

Use the scaled balanced logistic regression model for this because `StandardScaler` puts numeric features on a comparable scale. Larger positive coefficients indicate features that increase predicted purchase probability more strongly.

Important:

> Coefficients show association with purchase probability, not causation.


In [ ]:
# Feature importance using scaled balanced logistic regression coefficients
feature_names = X.columns
scaled_lr_model = LR_scaled_balanced.named_steps['model']
coefficients = scaled_lr_model.coef_[0]

feature_importance = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'absolute_coefficient': np.abs(coefficients)
})

feature_importance = feature_importance.sort_values(
    'absolute_coefficient',
    ascending=False
).reset_index(drop=True)

feature_importance


In [ ]:
# Visualize logistic regression feature importance
plot_df = feature_importance.sort_values('coefficient')
colors = plot_df['coefficient'].apply(lambda x: '#54A24B' if x > 0 else '#E45756')

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(
    plot_df['feature'],
    plot_df['coefficient'],
    color=colors
)

ax.set_title('Feature Importance - Scaled Balanced Logistic Regression')
ax.set_xlabel('Coefficient value')
ax.set_ylabel('Feature')
ax.axvline(0, color='black', linewidth=1)

for bar, value in zip(bars, plot_df['coefficient']):
    ax.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f' {value:.3f}' if value >= 0 else f' {value:.3f}',
        va='center',
        ha='left' if value >= 0 else 'right',
        fontsize=9
    )

plt.tight_layout()
plt.show()


### Feature importance interpretation

`session_duration_mins` is the strongest contributor to predicted purchase probability.

This means longer sessions increase the model's predicted probability of purchase. This aligns with the earlier analytics where purchase sessions had much longer durations than non-purchase sessions.

`carted` is the second strongest positive contributor.

This means sessions with cart activity are more likely to be purchase sessions. This also matches the funnel analysis, where cart activity was much more common in purchase sessions.

`viewed` has a negative coefficient.

This may look surprising, but it makes sense in this dataset. Almost all sessions contain views, and many viewed sessions are `view_only` sessions with no purchase. After controlling for cart activity and session duration, simply having a view is not a strong purchase signal.

Overall interpretation:

> The model is mostly driven by deeper engagement signals: longer session duration and cart activity. Simple viewing behavior is not enough to predict purchase because most viewed sessions do not convert.

Business takeaway:

> To increase purchase probability, the business should focus on moving users beyond passive viewing into deeper engagement and cart activity.


Interpretation guide:

- Positive coefficients increase the predicted probability of purchase.
- Negative coefficients decrease the predicted probability of purchase.
- Larger absolute coefficient values indicate stronger contribution to the model prediction.

Expected business interpretation:

> Features such as cart activity and longer session duration are likely to contribute strongly to purchase probability because they represent deeper user engagement.

Important caveat:

This is a baseline completed-session model. Feature importance explains which completed-session behaviors distinguish purchase sessions from non-purchase sessions. It should not be interpreted as real-time causal impact.


## Feature Ablation: Remove viewed

The feature importance analysis showed that `viewed` has a negative coefficient and contributes less than `session_duration_mins` and `carted`.

This does not mean views are bad. It likely means that viewing is very common across sessions and many viewed sessions do not purchase.

To test whether `viewed` is useful, train another scaled balanced logistic regression model without the `viewed` feature.


In [ ]:
# Feature ablation: remove viewed feature
X_no_view = funnel_df[['carted', 'session_duration_mins']]
y_no_view = funnel_df['purchased']

xtrain_no_view, xtest_no_view, ytrain_no_view, ytest_no_view = train_test_split(
    X_no_view,
    y_no_view,
    test_size=0.2,
    random_state=15
)

LR_no_view = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000))
])

LR_no_view.fit(xtrain_no_view, ytrain_no_view)

ypredtest_no_view = LR_no_view.predict(xtest_no_view)
yprobtest_no_view = LR_no_view.predict_proba(xtest_no_view)


In [ ]:
# Evaluation metrics for model without viewed
no_view_accuracy = accuracy_score(ytest_no_view, ypredtest_no_view)
no_view_conf_matrix = confusion_matrix(ytest_no_view, ypredtest_no_view)
no_view_precision = precision_score(ytest_no_view, ypredtest_no_view)
no_view_recall = recall_score(ytest_no_view, ypredtest_no_view)
no_view_class_report = classification_report(ytest_no_view, ypredtest_no_view)
no_view_roc_auc = roc_auc_score(ytest_no_view, yprobtest_no_view[:, 1])
no_view_loss = log_loss(ytest_no_view, yprobtest_no_view)


In [ ]:
# Print model-without-viewed results
print(f"Accuracy: {no_view_accuracy:.4f}")
print(f"ROC-AUC Score: {no_view_roc_auc:.4f}")
print(f"Log Loss: {no_view_loss:.4f}")
print("\nConfusion Matrix:\n", no_view_conf_matrix)
print(f"Precision: {no_view_precision:.4f}")
print(f"Recall:    {no_view_recall:.4f}")
print("\nClassification Report:\n", no_view_class_report)


In [ ]:
# Compare scaled balanced model with and without viewed feature
ablation_comparison = pd.DataFrame({
    'model': ['Scaled Balanced LR', 'Scaled Balanced LR without viewed'],
    'features': [
        'viewed, carted, session_duration_mins',
        'carted, session_duration_mins'
    ],
    'accuracy': [scaled_balanced_accuracy, no_view_accuracy],
    'roc_auc': [scaled_balanced_roc_auc, no_view_roc_auc],
    'log_loss': [scaled_balanced_loss, no_view_loss],
    'precision_purchase': [scaled_balanced_precision, no_view_precision],
    'recall_purchase': [scaled_balanced_recall, no_view_recall],
    'true_positives': [scaled_balanced_conf_matrix[1, 1], no_view_conf_matrix[1, 1]],
    'false_positives': [scaled_balanced_conf_matrix[0, 1], no_view_conf_matrix[0, 1]],
    'false_negatives': [scaled_balanced_conf_matrix[1, 0], no_view_conf_matrix[1, 0]]
})

ablation_comparison_display = ablation_comparison.copy()

for col in ['accuracy', 'roc_auc', 'precision_purchase', 'recall_purchase']:
    ablation_comparison_display[col] = ablation_comparison_display[col].map(lambda x: f'{x:.2%}')

ablation_comparison_display['log_loss'] = ablation_comparison_display['log_loss'].map(lambda x: f'{x:.4f}')

ablation_comparison_display


Interpretation guide:

If model performance stays the same or improves after removing `viewed`, then `viewed` can be removed from the final baseline model. A simpler model is preferred when it performs similarly.

If performance drops meaningfully, then `viewed` still adds useful signal and should be kept.


In [ ]:
X.head()

## Enhanced Baseline Logistic Regression

The enhanced model uses the Tableau-ready session-level dataset created in `Notebooks/Tableau Data Preparation.ipynb`.

That notebook adds reusable reporting and modeling features such as:

- `log_session_duration_mins`
- `visitor_session_count`
- `is_repeat_visitor`
- `session_start_hour`
- `session_start_dayofweek`
- `is_weekend`

This keeps the Logistic Regression notebook focused on modeling, evaluation, feature interpretation, ablation, and threshold tuning.

In [ ]:
# Feature set for enhanced baseline model
feature_cols_enhanced = [
    'viewed',
    'carted',
    'log_session_duration_mins',
    'visitor_session_count',
    'is_repeat_visitor',
    'session_start_hour',
    'session_start_dayofweek',
    'is_weekend'
]

X_enhanced = funnel_df[feature_cols_enhanced]
y_enhanced = funnel_df['purchased']

X_enhanced.head()

In [ ]:
# Confirm enhanced model features are available
funnel_df[feature_cols_enhanced + ['purchased']].isnull().sum()

In [ ]:
# Train/test split for enhanced baseline model
xtrain_enhanced, xtest_enhanced, ytrain_enhanced, ytest_enhanced = train_test_split(
    X_enhanced,
    y_enhanced,
    test_size=0.2,
    random_state=15,
    stratify=y_enhanced
)


In [ ]:
# Build enhanced scaled balanced logistic regression model
LR_enhanced = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000))
])

LR_enhanced.fit(xtrain_enhanced, ytrain_enhanced)

# Training predictions
ypredtrain_enhanced = LR_enhanced.predict(xtrain_enhanced)

# Testing predictions
ypredtest_enhanced = LR_enhanced.predict(xtest_enhanced)
yprobtest_enhanced = LR_enhanced.predict_proba(xtest_enhanced)


In [ ]:
# Accuracy comparison for enhanced model
print("Enhanced Logistic Regression - Training accuracy")
print("------------")
print(accuracy_score(ytrain_enhanced, ypredtrain_enhanced))

print("Enhanced Logistic Regression - Testing accuracy")
print("------------")
print(accuracy_score(ytest_enhanced, ypredtest_enhanced))


In [ ]:
# Evaluation metrics for enhanced logistic regression
enhanced_accuracy = accuracy_score(ytest_enhanced, ypredtest_enhanced)
enhanced_conf_matrix = confusion_matrix(ytest_enhanced, ypredtest_enhanced)
enhanced_precision = precision_score(ytest_enhanced, ypredtest_enhanced)
enhanced_recall = recall_score(ytest_enhanced, ypredtest_enhanced)
enhanced_class_report = classification_report(ytest_enhanced, ypredtest_enhanced)
enhanced_roc_auc = roc_auc_score(ytest_enhanced, yprobtest_enhanced[:, 1])
enhanced_loss = log_loss(ytest_enhanced, yprobtest_enhanced)


In [ ]:
# Print enhanced model results
print(f"Accuracy: {enhanced_accuracy:.4f}")
print(f"ROC-AUC Score: {enhanced_roc_auc:.4f}")
print(f"Log Loss: {enhanced_loss:.4f}")
print("\nConfusion Matrix:\n", enhanced_conf_matrix)
print(f"Precision: {enhanced_precision:.4f}")
print(f"Recall:    {enhanced_recall:.4f}")
print("\nClassification Report:\n", enhanced_class_report)


In [ ]:
# Compare current scaled balanced model with enhanced baseline model
# Note: the enhanced model uses a stratified split, so compare metrics directionally.
enhanced_model_comparison = pd.DataFrame({
    'model': ['Scaled Balanced LR', 'Enhanced Scaled Balanced LR'],
    'features': [
        'viewed, carted, session_duration_mins',
        ', '.join(feature_cols_enhanced)
    ],
    'accuracy': [scaled_balanced_accuracy, enhanced_accuracy],
    'roc_auc': [scaled_balanced_roc_auc, enhanced_roc_auc],
    'log_loss': [scaled_balanced_loss, enhanced_loss],
    'precision_purchase': [scaled_balanced_precision, enhanced_precision],
    'recall_purchase': [scaled_balanced_recall, enhanced_recall],
    'true_positives': [scaled_balanced_conf_matrix[1, 1], enhanced_conf_matrix[1, 1]],
    'false_positives': [scaled_balanced_conf_matrix[0, 1], enhanced_conf_matrix[0, 1]],
    'false_negatives': [scaled_balanced_conf_matrix[1, 0], enhanced_conf_matrix[1, 0]]
})

enhanced_model_comparison_display = enhanced_model_comparison.copy()

for col in ['accuracy', 'roc_auc', 'precision_purchase', 'recall_purchase']:
    enhanced_model_comparison_display[col] = enhanced_model_comparison_display[col].map(lambda x: f'{x:.2%}')

enhanced_model_comparison_display['log_loss'] = enhanced_model_comparison_display['log_loss'].map(lambda x: f'{x:.4f}')

enhanced_model_comparison_display


## Enhanced Model Feature Importance

Use the coefficients from the enhanced scaled balanced logistic regression model to understand which features contribute most to purchase prediction.

Because the enhanced model uses `StandardScaler`, coefficient magnitudes are more comparable across features.


In [ ]:
# Feature importance for enhanced scaled balanced logistic regression
feature_names_enhanced = X_enhanced.columns
enhanced_lr_model = LR_enhanced.named_steps['model']
enhanced_coefficients = enhanced_lr_model.coef_[0]

enhanced_feature_importance = pd.DataFrame({
    'feature': feature_names_enhanced,
    'coefficient': enhanced_coefficients,
    'absolute_coefficient': np.abs(enhanced_coefficients)
})

enhanced_feature_importance = enhanced_feature_importance.sort_values(
    'absolute_coefficient',
    ascending=False
).reset_index(drop=True)

enhanced_feature_importance


In [ ]:
# Visualize enhanced model feature importance
plot_df = enhanced_feature_importance.sort_values('coefficient')
colors = plot_df['coefficient'].apply(lambda x: '#54A24B' if x > 0 else '#E45756')

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(
    plot_df['feature'],
    plot_df['coefficient'],
    color=colors
)

ax.set_title('Feature Importance - Enhanced Scaled Balanced Logistic Regression')
ax.set_xlabel('Coefficient value')
ax.set_ylabel('Feature')
ax.axvline(0, color='black', linewidth=1)

for bar, value in zip(bars, plot_df['coefficient']):
    ax.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f' {value:.3f}',
        va='center',
        ha='left' if value >= 0 else 'right',
        fontsize=9
    )

plt.tight_layout()
plt.show()


Interpretation guide:

- Positive coefficients increase predicted purchase probability.
- Negative coefficients decrease predicted purchase probability.
- Larger absolute coefficients have stronger influence on the model.

Review whether the new engineered features, such as visitor frequency and session timing, contribute meaningful signal compared with the original features.


## Enhanced Model Feature Ablation

Instead of removing features only because their coefficients are negative, test whether removing them improves or hurts model performance.

Here we compare the full enhanced feature set against smaller feature sets:

- Full enhanced model
- Without weak time features
- Without `visitor_session_count`
- Without `viewed`
- Core intent features only

The best feature set should be chosen based on validation metrics such as ROC-AUC, PR-AUC, precision, recall, false positives, and false negatives.

In [ ]:
# Enhanced model feature ablation
from sklearn.metrics import average_precision_score

feature_sets_to_test = {
    'Full enhanced model': feature_cols_enhanced,
    'Without weak time features': [
        'viewed',
        'carted',
        'log_session_duration_mins',
        'visitor_session_count',
        'is_repeat_visitor'
    ],
    'Without visitor_session_count': [
        'viewed',
        'carted',
        'log_session_duration_mins',
        'is_repeat_visitor',
        'session_start_hour',
        'session_start_dayofweek',
        'is_weekend'
    ],
    'Without viewed': [
        'carted',
        'log_session_duration_mins',
        'visitor_session_count',
        'is_repeat_visitor',
        'session_start_hour',
        'session_start_dayofweek',
        'is_weekend'
    ],
    'Core intent features only': [
        'carted',
        'log_session_duration_mins',
        'is_repeat_visitor'
    ]
}

ablation_results = []

for model_name, feature_list in feature_sets_to_test.items():
    X_ablation = funnel_df[feature_list]
    y_ablation = funnel_df['purchased']

    X_train_ablation, X_test_ablation, y_train_ablation, y_test_ablation = train_test_split(
        X_ablation,
        y_ablation,
        test_size=0.2,
        random_state=42,
        stratify=y_ablation
    )

    ablation_model = Pipeline(steps=[
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ])

    ablation_model.fit(X_train_ablation, y_train_ablation)

    y_pred_ablation = ablation_model.predict(X_test_ablation)
    y_prob_ablation = ablation_model.predict_proba(X_test_ablation)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test_ablation, y_pred_ablation).ravel()

    ablation_results.append({
        'model': model_name,
        'features_used': ', '.join(feature_list),
        'feature_count': len(feature_list),
        'accuracy': accuracy_score(y_test_ablation, y_pred_ablation),
        'roc_auc': roc_auc_score(y_test_ablation, y_prob_ablation),
        'pr_auc': average_precision_score(y_test_ablation, y_prob_ablation),
        'log_loss': log_loss(y_test_ablation, y_prob_ablation),
        'precision_purchase': precision_score(y_test_ablation, y_pred_ablation),
        'recall_purchase': recall_score(y_test_ablation, y_pred_ablation),
        'true_positives': tp,
        'false_positives': fp,
        'false_negatives': fn
    })

ablation_comparison = pd.DataFrame(ablation_results)

ablation_comparison_display = ablation_comparison.copy()
percentage_cols = ['accuracy', 'roc_auc', 'pr_auc', 'precision_purchase', 'recall_purchase']
for col in percentage_cols:
    ablation_comparison_display[col] = (ablation_comparison_display[col] * 100).round(2).astype(str) + '%'

ablation_comparison_display['log_loss'] = ablation_comparison_display['log_loss'].round(4)
ablation_comparison_display

### Interpretation

Use this table to decide whether any features should be removed.

- If a smaller feature set keeps similar ROC-AUC, PR-AUC, and recall while reducing false positives, it may be better than the full enhanced model.
- If removing a feature lowers recall or PR-AUC meaningfully, that feature is still useful even if its coefficient is negative.
- `viewed` should only be removed if the comparison shows that the model performs equally well or better without it.
- `visitor_session_count` and `is_repeat_visitor` are related, so this comparison helps check whether both are needed.

For this project, prioritize recall and PR-AUC if the business goal is to identify likely purchase sessions, but also watch false positives because too many false positives can make marketing actions inefficient.

## Threshold Tuning for Enhanced Logistic Regression Model

The enhanced model currently uses the default threshold of `0.5`.

Because the dataset is highly imbalanced, threshold tuning helps choose a cutoff that better matches the business goal:

- lower threshold: catches more purchase sessions, but creates more false positives
- higher threshold: reduces false positives, but may miss more purchase sessions

For now, we will keep all enhanced features and tune only the classification threshold.

In [ ]:
# Threshold tuning for enhanced scaled balanced logistic regression
thresholds_to_test = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

enhanced_threshold_results = []

for threshold in thresholds_to_test:
    y_pred_threshold = (yprobtest_enhanced[:, 1] >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(ytest_enhanced, y_pred_threshold).ravel()

    enhanced_threshold_results.append({
        'threshold': threshold,
        'accuracy': accuracy_score(ytest_enhanced, y_pred_threshold),
        'precision_purchase': precision_score(ytest_enhanced, y_pred_threshold, zero_division=0),
        'recall_purchase': recall_score(ytest_enhanced, y_pred_threshold),
        'f1_score': f1_score(ytest_enhanced, y_pred_threshold),
        'true_positives': tp,
        'false_positives': fp,
        'false_negatives': fn
    })

enhanced_threshold_results = pd.DataFrame(enhanced_threshold_results)

enhanced_threshold_results_display = enhanced_threshold_results.copy()
for col in ['accuracy', 'precision_purchase', 'recall_purchase', 'f1_score']:
    enhanced_threshold_results_display[col] = (enhanced_threshold_results_display[col] * 100).round(2).astype(str) + '%'

enhanced_threshold_results_display

In [ ]:
# Visualize enhanced model threshold trade-off
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    enhanced_threshold_results['threshold'],
    enhanced_threshold_results['precision_purchase'],
    marker='o',
    label='Precision'
)
ax.plot(
    enhanced_threshold_results['threshold'],
    enhanced_threshold_results['recall_purchase'],
    marker='o',
    label='Recall'
)
ax.plot(
    enhanced_threshold_results['threshold'],
    enhanced_threshold_results['f1_score'],
    marker='o',
    label='F1-score'
)

ax.set_title('Enhanced Model Threshold Tuning')
ax.set_xlabel('Classification threshold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Select the threshold with the highest F1-score for the enhanced model
enhanced_best_f1_row = enhanced_threshold_results.loc[
    enhanced_threshold_results['f1_score'].idxmax()
]

enhanced_best_threshold_summary = pd.DataFrame({
    'metric': [
        'Best threshold by F1-score',
        'Precision at best threshold',
        'Recall at best threshold',
        'F1-score at best threshold',
        'True positives',
        'False positives',
        'False negatives'
    ],
    'value': [
        enhanced_best_f1_row['threshold'],
        f"{enhanced_best_f1_row['precision_purchase']:.2%}",
        f"{enhanced_best_f1_row['recall_purchase']:.2%}",
        f"{enhanced_best_f1_row['f1_score']:.2%}",
        int(enhanced_best_f1_row['true_positives']),
        int(enhanced_best_f1_row['false_positives']),
        int(enhanced_best_f1_row['false_negatives'])
    ]
})

enhanced_best_threshold_summary

### Interpretation

Threshold tuning does not change the model itself. It changes how strict the model is before labeling a session as a predicted purchase.

- If the threshold is low, the model predicts more purchase sessions. Recall usually increases, but false positives also increase.
- If the threshold is high, the model predicts fewer purchase sessions. Precision usually improves, but recall may decrease.
- The best F1 threshold gives a balanced trade-off between precision and recall, but the final threshold should depend on the business use case.

For this project, a higher threshold may be useful if we want fewer false positives before taking a marketing action. A lower threshold may be useful if missing potential purchase sessions is more costly.

Interpretation guide:

Compare the enhanced model with the scaled balanced baseline model.

Look for:

- higher ROC-AUC
- higher PR-AUC later, if evaluated
- better precision-recall balance
- fewer false positives without losing too many true positives
- lower log loss

If the enhanced model improves metrics meaningfully, the added visitor-frequency and time-based features provide useful predictive signal. If the results are similar, the simpler model may be preferred.
